In [1]:
# Importing the necessary libraries
import torch  # PyTorch
import torchvision.transforms as transforms  # image transformation
from torchvision.models import resnet50, ResNet50_Weights  # Importing the ResNet50 model
from PIL import Image  # image processing library
import psycopg2  # PostgreSQL database adapter

In [3]:
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V1  # Alternatively, use ResNet50_Weights.DEFAULT for the latest weights
    model = resnet50(weights=weights)  # Load the model with specified weights

    # Remove the last fully connected layer to get 2048-dimensional outputs
    model = torch.nn.Sequential(*list(model.children())[:-1])  # Keep all layers except the last one
    model.eval()  # Set the model to evaluation mode
    return model

In [4]:
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize(256),  # Resizing images
        transforms.CenterCrop(224),  # Center Cropped Image
        transforms.ToTensor(),  # Converting images to tensors
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # standardization
    ])

    image = Image.open(image_path).convert("RGB")  # Read images and convert to RGB format
    image_tensor = transform(image).unsqueeze(0)  # Add Batch Dimension
    return image_tensor

In [5]:
def generate_image_vector(model, image_path):
    image_tensor = preprocess_image(image_path)  # Preprocessed images
    with torch.no_grad():  # Disable gradient calculation
        vector = model(image_tensor).numpy()  # Generate vectors using models and convert to numpy arrays
    return vector.flatten()  # Flatten vectors for easy storage

In [ ]:
def search_similar_images(connection, input_vector, top_k):
    vector_str = ','.join(map(str, input_vector))  # Create a string of comma-separated values
    # print(vector_str)
    vector_str = f'[{vector_str}]'  # Format the string as a list

    query = """
    SELECT i.image_name,i.image_embedding <=> %s AS distance
    FROM image_info_test i
    ORDER BY distance
    LIMIT %s;
    """
    
    with connection.cursor() as cursor:
        cursor.execute(query,(vector_str,top_k))  # Execute the query
        results = cursor.fetchall()
    # return results
    # Extract and return only the image names from the results
    image_names = [result[0] for result in results]
    return image_names

In [7]:
def find_similar_images(image_path, model, connection, top_k):
    input_vector = generate_image_vector(model, image_path)  # Generate image vectors
    similar_images = search_similar_images(connection, input_vector, top_k)
    return similar_images

In [13]:
# main search function

TOP_K = 10 # Number of similar images to retrieve

image_path = "/Users/Tommy/AI/fashionCLIP/clothing-images/061/0619350002.jpg"

# Connecting to a PostgreSQL Database
connection = psycopg2.connect(
    dbname='image-search',  # database name
    user='postgres',  # user ID
    password='test-postgres',  # cryptographic
    host='127.0.0.1',  # RDSTerminal node of the instance
    port='5432'  # PostgreSQL ports
)

model = load_model()  # Load the ResNet50 model

similar_images = find_similar_images(image_path, model, connection, TOP_K)

print(f"Top {TOP_K} similar images: {similar_images}")

# for image_name, distance in similar_images:
#     print(f"Image: {image_name}, Distance: {distance}")

connection.close()  # Close the database connection

Top 10 similar images: ['0619350002.jpg', '0619350001.jpg', '0661578001.jpg', '0763727007.jpg', '0632307010.jpg', '0622381002.jpg', '0702623002.jpg', '0651869003.jpg', '0714556002.jpg', '0804506002.jpg']


In [ ]:
 prcision: 60%
['0495733003.jpg', v
 '0560030004.jpg', v
 '0560030005.jpg', 
 '0544054005.jpg', v
 '0503502003.jpg', v
 '0560030017.jpg', 
 '0559956011.jpg', v
 '0497787004.jpg', v
 '0634013002.jpg', v
 '0634013014.jpg']